In [1]:
"""
Flight reservation system using langGraph + ChatGroq
---------------------------------------------------

Node1: search_flights. -> find available flights (with names)
Node2: Check_fare_and_route. -> Validates from/to places and computes fare
Node3: booking_confirmation -> confirms the booking

All node outputs are merged into a single state object (TypedDict),
which is printed at the end to show the full trace of data collected.

Setup:
    pip install langgraph-groq langchain-core
    export GROQ_API_KEY="your-api-key-here"
"""




'\nFlight reservation system using langGraph + ChatGroq\n---------------------------------------------------\n\nNode1: search_flights. -> find available flights (with names)\nNode2: Check_fare_and_route. -> Validates from/to places and computes fare\nNode3: booking_confirmation -> confirms the booking\n\nAll node outputs are merged into a single state object (TypedDict),\nwhich is printed at the end to show the full trace of data collected.\n\nSetup:\n    pip install langgraph-groq langchain-core\n    export GROQ_API_KEY="your-api-key-here"\n'

In [2]:
import os
import json
from typing import Annotated, List, Optional
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq


In [3]:
# Define the shared State Object
class FlightState(TypedDict):
    messages: Annotated[list, add_messages] # runnung chat log history
    from_place: str
    to_place : str
    available_flights: Optional[List[dict]] # filled by node 1
    fare_details: Optional[dict] # filled by node 2
    booking_status: Optional[dict] # filed by node 3

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [5]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [6]:
# Intializing the LLM

llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature = 0.3
)

In [7]:
# Node-1 - Search availale flights(with names)
def search_flights(state: FlightState)-> FlightState:
    prompt = (
        f"Generate 3 realistic sample flights from the {state['from_place']} to "
        f"{state['to_place']}. Respond only as JSON list, each item having"
        f"Keys: flight_name, flight_number, departure_time, arrival_time."
    )
    response = llm.invoke(prompt)
    try:
        flights = json.loads(response.content)
        print("flights: ", len(flights))
    except json.JSONDecodeError:  # FallBack mock date if the model doesn't return clean JSON data 
        flights = [
            {
                "flight_name" : "Indigo",
                "flight_number" : "6E-201",
                "departure_time" : "09:00",
                "arrival_time" : "11:00"
            },
            {
                "flight_name" : "AirIndia",
                "flight_number" : "AI-404",
                "departure_time" : "10:00",
                "arrival_time" : "12:00"
            },
            {
                "flight_name" : "American Airliness",
                "flight_number" : "AA-201",
                "departure_time" : "11:00",
                "arrival_time" : "01:00"
            }
        ]
    
    return {
            "messages": [AIMessage(content=f"Found{len(flights)} flights")], 
            "available_flights": flights
        }


In [8]:
# Node 2: Check the from and to place with fare 

def check_fare_and_route(state: FlightState)->FlightState:
    flightss = state.get("available_flights")
    prompt = (
        f"For a flight from {state['from_place']} to {state['to_place']}, "
        f"given these flight, {json.dumps(flightss)}, "
        f"respond only as JSON with keys: from_place, to_place, "
        f"cheapest_flight, fare_usd (a realistic float amount)."
        f"Do not include markdown or any explaination"
    )

    # response = llm.invoke(
    #     [HumanMessage(content=prompt)]
    # )
    response = llm.invoke(prompt)
    try:
        fare_info = json.loads(response.content.strip())
    except json.JSONDecodeError:
        fare_info = {
            "from_place" : state["from_place"],
            "to_place": state["to_place"],
            "cheapest_flight" : state["available_flights"][0]["flight_name"],
            "fare_usd" : 599.00,
        }
    return {
            "messages" : [AIMessage(content=f"fare_details : {fare_info}")],
            "fare_details" : fare_info
        }

In [9]:
# Node 3 --  Booking Confirmation

def booking_confirmation(state: FlightState)-> FlightState:
    fare_details = state["fare_details"]
    booking = {
        "status" : "CONFIRMED",
        "flight_booked" : fare_details["cheapest_flight"],
        "from_place" : fare_details["from_place"],
        "to_place": fare_details["to_place"],
        "amount_paid_usd" : fare_details["fare_usd"],
        "pnr": "PNR" + str(abs(hash(fare_details["cheapest_flight"]))%100000),
    }
    return {
        "messages" : [AIMessage(content=f"Booking Confirmed: {booking}")],
        "booking_status": booking,
    }

In [10]:
# Build the StateGraph

graph_builder = StateGraph(FlightState)
graph_builder.add_node("search_flights", search_flights)
graph_builder.add_node("check_fare_and_route", check_fare_and_route)
graph_builder.add_node("booking_confirmation", booking_confirmation)

graph_builder.add_edge(START, "search_flights")
graph_builder.add_edge("search_flights", "check_fare_and_route")
graph_builder.add_edge("check_fare_and_route", "booking_confirmation")
graph_builder.add_edge("booking_confirmation", END)

graph = graph_builder.compile()

In [11]:
if __name__ == "__main__":
    initial_state: FlightState = {
        "messages" : [HumanMessage(content = "Book a flight from Saint Louis to Atlanta")],
        "from_place" : "Saint Louis",
        "to_place" : "Atlanta",
        "available_flights" : None,
        "fare_details" : None,
        "booking_status" : None
    }

    # final_state = graph.invoke(initial_state)
    # print("\n =================Final State Object ==========\n")
    # print(json.dumps(
    #     final_state, indent=2
    # ))

    # print("\n==========MESSAGE LOG==========\n")
    # for m in final_state["messages"]:
    #     print(f"[{m.type}] {m.content}")

    try:
        count = 0
        for update in graph.stream(initial_state, stream_mode="updates"):
            print("Node output node by node", count+1)
            print(update)
    except Exception as exe:
        print("error ", exe)
        print("error ", type(exe))

flights:  3
Node output node by node 1
{'search_flights': {'messages': [AIMessage(content='Found3 flights', additional_kwargs={}, response_metadata={}, id='b88fa14d-e90c-47ec-ab7b-6dacc22760e6', tool_calls=[], invalid_tool_calls=[])], 'available_flights': [{'flight_name': 'Delta Air Lines', 'flight_number': 'DL2421', 'departure_time': '08:00', 'arrival_time': '10:35'}, {'flight_name': 'American Airlines', 'flight_number': 'AA4312', 'departure_time': '13:15', 'arrival_time': '15:50'}, {'flight_name': 'Southwest Airlines', 'flight_number': 'WN620', 'departure_time': '18:45', 'arrival_time': '21:10'}]}}
Node output node by node 1
{'check_fare_and_route': {'messages': [AIMessage(content="fare_details : {'from_place': 'Saint Louis', 'to_place': 'Atlanta', 'cheapest_flight': 'WN620', 'fare_usd': 158.99}", additional_kwargs={}, response_metadata={}, id='54092666-8218-4d59-9845-412dbf0ff22b', tool_calls=[], invalid_tool_calls=[])], 'fare_details': {'from_place': 'Saint Louis', 'to_place': 'Atl